# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tracy030115/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Content Archetypes

The cluster names come from the original heuristic label in the analysis script. The clusters are at an earlier stage by inspecting cluster profiles and those names got carried into this report as historical labels rather than being freshly assigned based on this specific run's output.

The validation design does not carry the claim. Four of the five clusters, everything except the smallest one at 3% of content, all carry the identical label Rising Stars. A name meant to describe one recurring page type is not doing that job if it gets reused for four separate clusters with meaningfully different profiles. Cluster 3 averages 3.9K impressions while cluster 5 averages 2.3K. Cluster 2 averages 180 days old while cluster 3 averages 313 days old. This is close to the same failure I ran into with my own rising_stars archetype before I fixed the flipped condition, a name that sounded specific but was actually catching a broader, less coherent group than intended. The paper's own PCA plot backs this up, only 18% and 12% variance explained on the two shown components, with cluster overlap described as fuzzy rather than cleanly separated.

Feature Importance

The label comes from Random Forest feature importance, ranking which inputs the model leaned on most when predicting health score. Average position comes out at 43%, impressions at 32%, scroll depth at 15%. Placing this under a heading called What Predicts Health implies these are the real levers behind strong pages.

The validation design does not carry that claim. Health score is partly built from position and impressions in the first place, the composite score is impressions worth 30 points plus position worth 30 points plus CTR worth 20 points plus scroll depth worth 20 points. Finding that position and impressions are the top predictors of a score that already includes position and impressions as direct inputs is not external validation, it is closer to the model rediscovering its own formula. This matters for my own archetype work too, since champions and hidden_gems are both partly defined by is_good_position and is_high_volume, the same two dimensions this model is technically discovering.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


In [2]:
import pandas as pd
import numpy as np

page_level = con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    ),
    fact_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS total_impressions,
            SUM(gsc_clicks) AS total_clicks,
            AVG(gsc_avg_position) AS avg_position,
            SUM(ga4_sessions) AS total_sessions,
            SUM(ga4_engaged_sessions) AS total_engaged_sessions,
            BOOL_OR(gsc_data_available) AS gsc_data_available_any,
            BOOL_OR(ga4_data_available) AS ga4_data_available_any
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        GROUP BY client_hash_id, content_hash_id
    ),
    content_meta AS (
        SELECT
            content_hash_id,
            keyword_hash_id,
            content_updated_date,
            is_published,
            is_deleted
        FROM read_parquet('{rel}/dim_content.parquet')
        WHERE is_published = true AND is_deleted = false
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        cm.keyword_hash_id,
        f.total_impressions,
        f.total_clicks,
        f.avg_position,
        f.total_sessions,
        f.total_engaged_sessions,
        f.gsc_data_available_any,
        f.ga4_data_available_any,
        DATE_DIFF('day', cm.content_updated_date, ref.max_date) AS days_since_last_update,
        CASE WHEN f.total_impressions > 0 THEN f.total_clicks * 1.0 / f.total_impressions ELSE NULL END AS ctr,
        CASE WHEN f.total_sessions > 0 THEN f.total_engaged_sessions * 1.0 / f.total_sessions ELSE NULL END AS engagement_rate
    FROM fact_agg f
    JOIN content_meta cm ON f.content_hash_id = cm.content_hash_id
    CROSS JOIN ref
""").df()

print(page_level.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(394928, 13)


In [3]:
def position_bucket(pos):
    if pos is None or pd.isna(pos):
        return None
    if pos <= 3:
        return "1_pos_1-3"
    elif pos <= 10:
        return "2_pos_4-10"
    elif pos <= 20:
        return "3_pos_11-20"
    else:
        return "4_pos_21plus"

expected_ctr_by_bucket = {
    "1_pos_1-3": 0.027805,
    "2_pos_4-10": 0.004229,
    "3_pos_11-20": 0.004184,
    "4_pos_21plus": 0.002788,
}

page_level["position_bucket"] = page_level["avg_position"].apply(position_bucket)
page_level["expected_ctr"] = page_level["position_bucket"].map(expected_ctr_by_bucket)
page_level["ctr_gap"] = page_level["expected_ctr"] - page_level["ctr"]

In [4]:
keyword_counts = (
    page_level[page_level["total_impressions"] > 0]
    .groupby(["client_hash_id", "keyword_hash_id"])["content_hash_id"]
    .nunique()
    .reset_index(name="pages_ranking_for_keyword")
)
page_level = page_level.merge(keyword_counts, on=["client_hash_id", "keyword_hash_id"], how="left")
page_level["cannibalization_risk"] = page_level["pages_ranking_for_keyword"].fillna(0) > 1

In [5]:
MIN_IMPRESSIONS = 10
STALE_THRESHOLD_DAYS = 180
HIGH_VOLUME_IMPRESSIONS = page_level["total_impressions"].quantile(0.75)

def assign_archetype(row):
    gsc_available = bool(row["gsc_data_available_any"]) if pd.notna(row["gsc_data_available_any"]) else False
    if (not gsc_available) or row["total_impressions"] < MIN_IMPRESSIONS:
        return "INSUFFICIENT_DATA"

    if pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] < 0:
        return "INSUFFICIENT_DATA"

    is_good_position = pd.notna(row["avg_position"]) and row["avg_position"] <= 10
    is_high_volume = row["total_impressions"] >= HIGH_VOLUME_IMPRESSIONS
    is_low_ctr = pd.notna(row["ctr_gap"]) and row["ctr_gap"] > 0
    is_stale = pd.notna(row["days_since_last_update"]) and row["days_since_last_update"] > STALE_THRESHOLD_DAYS
    has_low_engagement = pd.notna(row["engagement_rate"]) and row["engagement_rate"] < 0.3
    is_no_demand = row["total_impressions"] < MIN_IMPRESSIONS * 3
    ga4_available = bool(row["ga4_data_available_any"]) if pd.notna(row["ga4_data_available_any"]) else False
    is_cannibalization = bool(row["cannibalization_risk"]) if pd.notna(row["cannibalization_risk"]) else False

    if is_cannibalization:
        return "cannibalization_risk"
    if is_good_position and is_high_volume and not is_low_ctr and not is_stale:
        return "champions"
    if is_good_position and is_high_volume and is_stale:
        return "stale_visible_pages"
    if ga4_available and is_good_position and is_high_volume and has_low_engagement:
        return "engagement_problem_pages"
    if is_good_position and not is_high_volume and not is_low_ctr:
        return "hidden_gems"
    if not is_good_position and pd.notna(row["ctr_gap"]) and row["ctr_gap"] < 0 and row["total_impressions"] >= HIGH_VOLUME_IMPRESSIONS * 0.25:
        return "rising_stars"
    if is_no_demand:
        return "weak_no_demand_pages"

    return "monitor"

page_level["archetype"] = page_level.apply(assign_archetype, axis=1)
page_level["archetype"].value_counts()

,count
archetype,
INSUFFICIENT_DATA,272549
monitor,56390
rising_stars,19210
weak_no_demand_pages,17426
champions,14300
engagement_problem_pages,12752
hidden_gems,2272
stale_visible_pages,27
cannibalization_risk,2


In [6]:
feature_cols = ["total_impressions", "total_clicks", "ctr", "avg_position", "days_since_last_update"]

model_data = page_level[page_level["archetype"] != "INSUFFICIENT_DATA"].copy()
model_data = model_data.dropna(subset=feature_cols)
print(f"model_data rows: {len(model_data)}")

model_data rows: 122379


In [7]:
np.random.seed(42)
all_clients = page_level["client_hash_id"].unique()
np.random.shuffle(all_clients)

n_test_clients = max(1, int(len(all_clients) * 0.2))
test_clients = set(all_clients[:n_test_clients])
train_clients = set(all_clients[n_test_clients:])

model_data["split"] = np.where(model_data["client_hash_id"].isin(train_clients), "train", "test")

train = model_data[model_data["split"] == "train"].copy()
test = model_data[model_data["split"] == "test"].copy()
print(f"train rows: {len(train)}, test rows: {len(test)}")

train rows: 102269, test rows: 20110


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test = scaler.transform(test[feature_cols])

k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
train_clusters = kmeans.fit_predict(X_train)
test_clusters = kmeans.predict(X_test)

train["kmeans_cluster"] = train_clusters
test["kmeans_cluster"] = test_clusters

train_silhouette = silhouette_score(X_train, train_clusters)
test_silhouette = silhouette_score(X_test, test_clusters)

print(f"GROUPED split, train silhouette: {train_silhouette:.3f}")
print(f"GROUPED split, test silhouette: {test_silhouette:.3f}")

GROUPED split, train silhouette: 0.483
GROUPED split, test silhouette: 0.491


In [9]:
from sklearn.model_selection import train_test_split

naive_train, naive_test = train_test_split(model_data, test_size=0.2, random_state=42)

scaler_naive = StandardScaler()
X_train_naive = scaler_naive.fit_transform(naive_train[feature_cols])
X_test_naive = scaler_naive.transform(naive_test[feature_cols])

kmeans_naive = KMeans(n_clusters=7, random_state=42, n_init=10)
naive_train_clusters = kmeans_naive.fit_predict(X_train_naive)
naive_test_clusters = kmeans_naive.predict(X_test_naive)

naive_train_silhouette = silhouette_score(X_train_naive, naive_train_clusters)
naive_test_silhouette = silhouette_score(X_test_naive, naive_test_clusters)

print(f"NAIVE split, train silhouette: {naive_train_silhouette:.3f}")
print(f"NAIVE split, test silhouette: {naive_test_silhouette:.3f}")

NAIVE split, train silhouette: 0.495
NAIVE split, test silhouette: 0.495


In [10]:
comparison_before_after = pd.DataFrame({
    "split_type": ["Naive random row split", "Grouped by client split"],
    "train_silhouette": [round(naive_train_silhouette, 3), round(train_silhouette, 3)],
    "test_silhouette": [round(naive_test_silhouette, 3), round(test_silhouette, 3)],
    "train_test_gap": [
        round(naive_train_silhouette - naive_test_silhouette, 3),
        round(train_silhouette - test_silhouette, 3)
    ],
})
comparison_before_after

,split_type,train_silhouette,test_silhouette,train_test_gap
0,Naive random row split,0.495,0.495,-0.000
1,Grouped by client split,0.483,0.491,-0.009


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 1. Confirm no future-window leakage in the staleness reference date
con.sql(f"""
    SELECT MAX(report_date) AS max_report_date, MIN(report_date) AS min_report_date
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,max_report_date,min_report_date
0,2026-06-30,2026-06-01


In [12]:
# 2. Confirm content_updated_date doesn't reach meaningfully past the fact table's
con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    )
    SELECT COUNT(*) AS rows_updated_after_max_report_date
    FROM read_parquet('{rel}/dim_content.parquet') dc
    CROSS JOIN ref
    WHERE dc.content_updated_date > ref.max_date
""").df()

,rows_updated_after_max_report_date
0,70305


In [13]:
model_data_content_ids = set(model_data["content_hash_id"])

con.sql(f"""
    WITH ref AS (
        SELECT MAX(report_date) AS max_date
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    )
    SELECT COUNT(*) AS rows_updated_after_max_report_date
    FROM read_parquet('{rel}/dim_content.parquet') dc
    CROSS JOIN ref
    WHERE dc.content_updated_date > ref.max_date
    AND dc.content_hash_id IN {tuple(model_data_content_ids) if len(model_data_content_ids) > 1 else "('" + list(model_data_content_ids)[0] + "')"}
""").df()

,rows_updated_after_max_report_date
0,0


In [14]:
# 3. Confirm the negative-days fix actually held
print(f"negative days_since_last_update rows in model_data: {(model_data['days_since_last_update'] < 0).sum()}")

negative days_since_last_update rows in model_data: 0


In [15]:
# 4. Confirm no product flag or precomputed tier leaked into the six modeling features
used_features = set(feature_cols)

fact_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')").df()["column_name"].tolist()
dim_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()["column_name"].tolist()

print("Fact columns NOT in feature_cols:", [c for c in fact_cols if c not in used_features])
print("Dim columns NOT in feature_cols:", [c for c in dim_cols if c not in used_features])

Fact columns NOT in feature_cols: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Dim columns NOT in feature_cols: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_opti

In [16]:
# 5. Confirm the client grouping actually held, no client appears in both train and test
overlap_clients = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"clients appearing in both train and test: {len(overlap_clients)}")

clients appearing in both train and test: 0


In [17]:
# 6. Confirm the archetype label itself
print("archetype in feature_cols:", "archetype" in feature_cols)
print("cannibalization_risk in feature_cols:", "cannibalization_risk" in feature_cols)
print("keyword_hash_id in feature_cols:", "keyword_hash_id" in feature_cols)

archetype in feature_cols: False
cannibalization_risk in feature_cols: False
keyword_hash_id in feature_cols: False


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The naive split's train and test silhouette came out nearly identical, 0.495 versus 0.495, but that is not evidence of generalization. The split design let the same client's pages sit on both sides, so a close match here does not tell me much. The grouped split, 0.483 train and 0.491 test, is the measured result that actually supports a generalization claim, and it is directional evidence, not proof the archetypes will hold on entirely new clients going forward.

Six specific checks were run against the final feature set and split, and each came back with the expected result, zero negative-days rows, zero client overlap across train and test, no product flag columns among the modeling inputs. This is decision-support evidence that the checked leakage paths are closed.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.